<a href="https://colab.research.google.com/github/fattah-abdullah/ProyekAkhir_5A_CardiovascularDisease/blob/main/P12B_Varian2_Perlakuan_Data_Mifa_Miftahul_Falaah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Varian 2 — Perlakuan Data
## Proyek Akhir Pembelajaran Mesin (INF62325) — Cabang A: Regresi Linear

| | |
|---|---|
| Kelompok | 6 |
| Anggota penanggung jawab | Mifa Miftahul Falaah, 2488010072 |

---

### Panduan notebook ini

**Notebook ini milik Anda sendiri.** Dua anggota lain mengerjakan varian mereka
di notebook terpisah, boleh bersamaan. Yang Anda hasilkan dibandingkan dengan
milik mereka di Pertemuan 13.

**Masukan.** `p12_data.joblib` dari notebook bersama `P12_Data_Bersama.ipynb`.
Bila berkas itu belum ada, notebook ini berhenti di sel pemeriksaan.

**Varian Anda: pencilan dibuang dengan IQR, lalu fitur diskalakan**

Fitur yang dipakai sama dengan Varian 1. Yang berbeda: baris yang
nilainya di luar batas IQR dibuang **dari data latih saja**, lalu
fitur numerik diskalakan di dalam `Pipeline`.

Anda tidak menentukan angka apa pun — batas IQR dihitung otomatis.
Yang Anda catat adalah **berapa baris yang dibuang**, karena itu yang
akan ditanyakan.

**Luaran.** `varian2.joblib` (model terlatih) dan `varian2_hasil.json`
(angka evaluasinya).

**Satu aturan yang tidak boleh dilanggar.** Jangan menjalankan
`train_test_split` di notebook ini. Pembagian data sudah dilakukan sekali di
notebook bersama, dan notebook ini memeriksanya lewat sidik jari. Varian
1 dan 3 diuji pada data uji yang sama; begitu salah satu
membaginya sendiri, perbandingan di Pertemuan 13 batal.

**Perkiraan waktu.** 60 menit.

---
## Langkah A — Samakan versi scikit-learn (WAJIB, setiap kali membuka notebook ini)

Colab memulai sesi baru setiap kali notebook dibuka, jadi langkah ini diulang
**di setiap pertemuan**, bukan sekali saja.

Gunanya menyamakan versi scikit-learn di Colab dengan versi yang dipasang
Streamlit Community Cloud. Tanpa ini, model berhasil dilatih tetapi gagal dimuat
di aplikasi web dengan galat
`AttributeError: ... has no attribute '_RemainderColsList'`.

In [14]:
!pip install -q scikit-learn==1.9.1

### Restart dulu, jangan dilewati

Pilih menu **Runtime → Restart session**, lalu jalankan ulang dari sel paling atas.
Tanpa restart, Python masih memakai scikit-learn versi lama yang sudah terlanjur
dimuat ke memori, dan kesalahannya tidak terlihat sampai tahap paling akhir.

In [15]:
import sklearn

print("scikit-learn:", sklearn.__version__)
assert sklearn.__version__ == "1.9.1", (
    "Versi belum sesuai. Jalankan sel pip di atas, lalu Runtime -> Restart session, "
    "lalu jalankan ulang dari sel paling atas."
)
print("Versi sudah sesuai.")

scikit-learn: 1.9.1
Versi sudah sesuai.


---
## Langkah B — Hubungkan Google Drive

Setiap pertemuan menyimpan hasilnya ke satu folder Drive yang sama, lalu pertemuan
berikutnya membacanya dari sana. Dengan begitu pekerjaan tidak hilang saat sesi Colab
berakhir, dan tidak perlu mengulang dari awal tiap minggu.

Folder yang dipakai: `/content/drive/MyDrive/ProyekAkhirML_RegresiLinear`

Saat sel di bawah dijalankan, Colab meminta izin mengakses Drive. Pilih akun Google
kelompok, lalu setujui.

In [16]:
import os
from google.colab import drive

drive.mount("/content/drive")

FOLDER = "/content/drive/MyDrive/ProyekAkhirML_RegresiLinear"
os.makedirs(FOLDER, exist_ok=True)

print("Folder proyek:", FOLDER)
print("Isi folder   :", sorted(os.listdir(FOLDER)) or "(masih kosong)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder proyek: /content/drive/MyDrive/ProyekAkhirML_RegresiLinear
Isi folder   : ['data_mentah.csv', 'model_terpilih.joblib', 'p12_data.joblib', 'p13_metrik.json', 'varian1.joblib', 'varian1_hasil.json', 'varian2.joblib', 'varian2_hasil.json', 'varian3.joblib', 'varian3_hasil.json']


---
## Langkah C — Muat data bersama

In [17]:
# Memastikan hasil notebook bersama P12_Data_Bersama benar-benar ada sebelum lanjut.
wajib = ["p12_data.joblib"]
hilang = [b for b in wajib if not os.path.exists(os.path.join(FOLDER, b))]

assert not hilang, (
    f"Berkas berikut belum ada di folder Drive: {hilang}. "
    "Jalankan notebook notebook bersama P12_Data_Bersama sampai selesai lebih dulu."
)
print("Semua berkas masukan tersedia:", wajib)

Semua berkas masukan tersedia: ['p12_data.joblib']


In [18]:
import joblib
import json
import numpy as np
import pandas as pd

p12 = joblib.load(os.path.join(FOLDER, "p12_data.joblib"))

X_latih = p12["X_latih"]; X_uji = p12["X_uji"]
y_latih = p12["y_latih"]; y_uji = p12["y_uji"]
kolom_numerik = p12["kolom_numerik"]
kolom_kategorik = p12["kolom_kategorik"]
TARGET = p12["TARGET"]

print("Data latih:", X_latih.shape, "| Data uji:", X_uji.shape)
print("Numerik  :", kolom_numerik)
print("Kategorik:", kolom_kategorik)

Data latih: (55993, 10) | Data uji: (13999, 10)
Numerik  : ['ap_lo', 'height', 'weight', 'age']
Kategorik: ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']


### Pemeriksaan sidik jari

Sel ini membuktikan bahwa data uji Anda sama persis dengan data uji kedua
anggota lain. Kodenya harus sama dengan yang dicetak notebook bersama.

In [19]:
import hashlib

def sidik_jari(X_uji):
    """Sidik jari pembagian data uji.

    Dipakai ketiga notebook varian untuk membuktikan bahwa mereka benar-benar
    memakai data uji yang sama. Bila ada anggota yang menjalankan
    train_test_split sendiri, sidik jarinya berbeda dan notebook berhenti.
    """
    isi = "|".join(str(i) for i in sorted(X_uji.index.tolist()))
    isi += f"|{X_uji.shape[0]}x{X_uji.shape[1]}"
    return hashlib.sha256(isi.encode()).hexdigest()[:16]


sidik_sekarang = sidik_jari(X_uji)
print("Sidik jari data uji :", sidik_sekarang)
print("Tercatat di P12     :", p12["sidik_jari_uji"])

assert sidik_sekarang == p12["sidik_jari_uji"], (
    "Data uji TIDAK sama dengan yang dibuat di P12 bagian bersama.\n"
    "Jangan menjalankan train_test_split di notebook ini. Muat ulang\n"
    "p12_data.joblib apa adanya, lalu jalankan dari sel paling atas."
)
print("\nCocok. Varian 2 diuji pada data uji yang sama dengan varian lain.")

Sidik jari data uji : 3e147073c4769645
Tercatat di P12     : 3e147073c4769645

Cocok. Varian 2 diuji pada data uji yang sama dengan varian lain.


---
## Langkah D — Merakit Pipeline Varian 2

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Menghitung batas IQR

Batas dihitung **dari data latih saja**. Memakai seluruh data untuk menghitung
batas berarti informasi dari data uji ikut menentukan baris mana yang dibuang —
bentuk kebocoran data yang sama seperti menukar urutan `train_test_split`.

In [21]:
Q1 = X_latih[kolom_numerik].quantile(0.25)
Q3 = X_latih[kolom_numerik].quantile(0.75)
IQR = Q3 - Q1
batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

print("Batas bawah:\n", batas_bawah.round(3).to_string())
print("\nBatas atas:\n", batas_atas.round(3).to_string())

Batas bawah:
 ap_lo        65.0
height      142.5
weight       39.5
age       12138.0

Batas atas:
 ap_lo       105.0
height      186.5
weight      107.5
age       26834.0


### Membuang baris pencilan dari data latih

Perhatikan bahwa `X_uji` tidak disentuh. Membuang pencilan dari data uji akan
membuat angka evaluasi terlihat bagus karena kasus sulitnya sudah dihapus —
bukan karena modelnya lebih baik. Sel di bawah mencetak jumlah baris uji
sebagai bukti bahwa jumlahnya tidak berubah.

In [22]:
masker = ~((X_latih[kolom_numerik] < batas_bawah) |
           (X_latih[kolom_numerik] > batas_atas)).any(axis=1)

X_latih_v2 = X_latih[masker]
y_latih_v2 = y_latih[masker]

DIBUANG = int((~masker).sum())
PERSEN = DIBUANG / len(X_latih) * 100

print(f"Baris latih: {len(X_latih)} -> {len(X_latih_v2)} "
      f"({DIBUANG} baris dibuang, {PERSEN:.1f}%)")
print(f"Baris uji  : {len(X_uji)} (TIDAK disentuh)")

if PERSEN >= 25:
    print("\nPERHATIAN: lebih dari seperempat data latih terbuang.")
    print("Kemungkinan sebaran datanya memang lebar dan nilai-nilai itu bukan")
    print("kesalahan pencatatan. Tanyakan ke dosen sebelum melanjutkan.")

Baris latih: 55993 -> 50616 (5377 baris dibuang, 9.6%)
Baris uji  : 13999 (TIDAK disentuh)


### Merakit Pipeline

Perhatikan bahwa pembuangan baris tadi **tidak** dimasukkan ke dalam `Pipeline`,
sedangkan penskalaan dimasukkan. Alasannya: isi `Pipeline` dikenakan pada setiap
masukan, termasuk satu baris yang nanti diketik pengguna di aplikasi web.
Membuang baris adalah keputusan tentang *data latih mana yang dipakai*, bukan
transformasi atas masukan. Aplikasi web tidak pernah membuang baris.

In [23]:
fitur_numerik = kolom_numerik
fitur_kategorik = []            # Varian 2 tidak memakai kolom kategorik

pipeline = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([
            ("imputasi", SimpleImputer(strategy="median")),
            ("skala", StandardScaler()),
        ]), fitur_numerik),
    ])),
    ("reg", LinearRegression()),
])

print("Fitur yang dipakai :", fitur_numerik)
print("Baris latih        :", len(X_latih_v2))
pipeline

Fitur yang dipakai : ['ap_lo', 'height', 'weight', 'age']
Baris latih        : 50616


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

---
## Langkah E — Melatih dan mengukur

Diukur pada `X_uji`, data uji bersama. Inilah angka yang masuk ke tabel
perbandingan Pertemuan 13.

In [24]:
pipeline.fit(X_latih_v2, y_latih_v2)

duga_uji = pipeline.predict(X_uji)
mae = mean_absolute_error(y_uji, duga_uji)
rmse = float(np.sqrt(mean_squared_error(y_uji, duga_uji)))
r2 = r2_score(y_uji, duga_uji)
r2_latih = r2_score(y_latih_v2, pipeline.predict(X_latih_v2))

print(f"Hasil Varian 2 pada DATA UJI")
print(f"  MAE  = {mae:,.2f}")
print(f"  RMSE = {rmse:,.2f}")
print(f"  R2   = {r2:.4f}")
print(f"\nR2 pada data latih = {r2_latih:.4f}  "
      f"(selisih {r2_latih - r2:+.4f})")
print("Selisih besar menandakan model menghafal data latihnya.")

Hasil Varian 2 pada DATA UJI
  MAE  = 30.15
  RMSE = 242.19
  R2   = -87.1445

R2 pada data latih = 0.0041  (selisih +87.1486)
Selisih besar menandakan model menghafal data latihnya.


### Validasi silang

In [25]:
skor = cross_val_score(pipeline, X_latih_v2, y_latih_v2, cv=5, scoring="r2")

cv_rata = float(skor.mean())
cv_simpangan = float(skor.std())
print(f"Validasi silang 5 lipatan Varian 2")
print(f"  rata-rata R2   = {cv_rata:.4f}")
print(f"  simpangan baku = {cv_simpangan:.4f}")
print(f"  tiap lipatan   = {skor.round(3)}")
print("\nSimpangan baku besar berarti angka tunggal pada data uji")
print("tidak boleh dipercaya sepenuhnya. Catat angka ini.")

Validasi silang 5 lipatan Varian 2
  rata-rata R2   = 0.0043
  simpangan baku = 0.0012
  tiap lipatan   = [0.005 0.005 0.003 0.005 0.003]

Simpangan baku besar berarti angka tunggal pada data uji
tidak boleh dipercaya sepenuhnya. Catat angka ini.


### Temuan Anda — ISI SEL INI

Ini yang Anda pertanggungjawabkan sendiri saat tanya jawab. Sebutkan angkanya.

- Fitur yang varian saya pakai:

  ap_lo, height, weight, age.
- Yang saya perlakukan berbeda dari varian lain:

  Menghapus pencilan menggunakan metode Interquartile Range (IQR) pada data latih dan melakukan penskalaan menggunakan StandardScaler.

- Hasil pada data uji — MAE, RMSE, R²:

  -MAE = 30,15

  -RMSE = 242,19

  -R² = -87,1445
- Kestabilan menurut validasi silang:

  Rata-rata R² = 0,0043 dengan simpangan baku 0,0012.
- Dugaan saya mengapa hasilnya begini:

  Nilai R² yang negatif menunjukkan bahwa model belum mampu memprediksi target dengan baik. Kemungkinan fitur yang digunakan belum cukup menjelaskan variasi tekanan darah sistolik (ap_hi).

Dugaan boleh salah. Pertemuan 13 yang akan membuktikannya, dan yang dinilai
adalah apakah Anda paham **apa** yang Anda ubah dan **mengapa** perubahan itu
bisa berpengaruh.

---
## Langkah F — Menyimpan hasil ke Drive

In [26]:
ANGGOTA = "(Mifa Miftahul Falaah, 2488010072)"      # ISI dengan nama dan NIM Anda

hasil = {
    "varian": 2,
    "nama": "Perlakuan Data",
    "anggota": ANGGOTA,
    "fitur_numerik": fitur_numerik,
    "fitur_kategorik": fitur_kategorik,
    "baris_latih": int(len(X_latih_v2)),
    "mae": float(mae),
    "rmse": float(rmse),
    "r2": float(r2),
    "r2_latih": float(r2_latih),
    "cv_rata": cv_rata,
    "cv_simpangan": cv_simpangan,
    "sidik_jari_uji": sidik_sekarang,
    "baris_dibuang": DIBUANG,
    "persen_dibuang": round(PERSEN, 2),
    "catatan": f"Pencilan IQR dibuang dari data latih ({DIBUANG} baris, {PERSEN:.1f}%), lalu fitur numerik diskalakan.",
}

assert ANGGOTA != "(nama, NIM)", (
    "Isi ANGGOTA dengan nama dan NIM Anda di baris paling atas sel ini,\n"
    "lalu jalankan ulang. Berkas sengaja belum ditulis: tanpa nama,\n"
    "Pertemuan 13 tidak bisa menyebut siapa pemilik varian ini."
)

joblib.dump(pipeline, os.path.join(FOLDER, "varian2.joblib"))
with open(os.path.join(FOLDER, "varian2_hasil.json"), "w", encoding="utf-8") as f:
    json.dump(hasil, f, indent=2, ensure_ascii=False)

print("Tersimpan: varian2.joblib dan varian2_hasil.json")
print(json.dumps(hasil, indent=2, ensure_ascii=False))
print("\nIsi folder:", sorted(os.listdir(FOLDER)))

Tersimpan: varian2.joblib dan varian2_hasil.json
{
  "varian": 2,
  "nama": "Perlakuan Data",
  "anggota": "(Mifa Miftahul Falaah, 2488010072)",
  "fitur_numerik": [
    "ap_lo",
    "height",
    "weight",
    "age"
  ],
  "fitur_kategorik": [],
  "baris_latih": 50616,
  "mae": 30.14577651438388,
  "rmse": 242.19215700464736,
  "r2": -87.14450139526238,
  "r2_latih": 0.004090533629702686,
  "cv_rata": 0.004265239480528394,
  "cv_simpangan": 0.0011633300067073224,
  "sidik_jari_uji": "3e147073c4769645",
  "baris_dibuang": 5377,
  "persen_dibuang": 9.6,
  "catatan": "Pencilan IQR dibuang dari data latih (5377 baris, 9.6%), lalu fitur numerik diskalakan."
}

Isi folder: ['data_mentah.csv', 'model_terpilih.joblib', 'p12_data.joblib', 'p13_metrik.json', 'varian1.joblib', 'varian1_hasil.json', 'varian2.joblib', 'varian2_hasil.json', 'varian3.joblib', 'varian3_hasil.json']


---
## Isi folder Drive setelah notebook ini

```
ProyekML_RegresiLinear/          ← folder Google Drive kelompok
├── data_mentah.csv              · dari bagian bersama
├── p12_data.joblib              · dari bagian bersama, DIBACA HARI INI
├── varian2.joblib              ← DIHASILKAN NOTEBOOK INI
└── varian2_hasil.json          ← DIHASILKAN NOTEBOOK INI
```

Dua berkas milik anggota lain (`varian1.*` dan `varian3.*`) muncul di folder
yang sama ketika mereka selesai. Ketiganya tidak saling menimpa.

## Daftar periksa

- [✔] Nama dan NIM terisi di sel identitas dan di `ANGGOTA`
- [✔] Sel sidik jari mencetak `Cocok`
- [✔] `train_test_split` **tidak** dijalankan di notebook ini
- [✔] Jumlah baris pencilan yang dibuang sudah dicatat
- [✔] Jumlah baris uji tetap sama seperti sebelum penyaringan
- [✔] Sel temuan terisi dengan angka
- [✔] Sel Langkah F mencetak `Tersimpan: varian2.joblib dan varian2_hasil.json`
- [✔] Notebook sudah tersimpan: **File → Save**

Pertemuan 13 baru bisa dijalankan setelah ketiga anggota menyelesaikan
notebooknya masing-masing.